<div class="align-center">
<a href="https://github.com/typedef-ai/fenic"><img src="https://github.com/typedef-ai/fenic/blob/main/docs/images/typedef-fenic-logo-github-yellow.png?raw=true" height="50"></a>
<a href="https://discord.gg/GdqF3J7huR"><img src="https://github.com/typedef-ai/fenic/blob/main/docs/images/join-the-discord.png?raw=true" height="50"></a>
<a href="https://docs.fenic.ai/latest/"><img src="https://github.com/typedef-ai/fenic/blob/main/docs/images/documentation.png?raw=true" height="50"></a>


</div>


# End-to-End LLM Feature Engineering for Recommendation Systems with Fenic and OpenAI's GPT-4 (Clustering, Tags, Recsys)

To run this notebook just press _"Run All"_ <span style="opacity:.8;">(in Google Colab: <b>Runtime ▸ Run all</b>)</span>

<p align="center">
  <a href="https://docs.fenic.ai">Read the Docs</a> •
  <a href="https://discord.com/invite/GdqF3J7huR">Join Discord</a> •
  <a href="https://github.com/typedef-ai/fenic">⭐️ Star fenic</a>
</p>

To install fenic locally, just follow the instructions on the [Github Repo](https://github.com/typedef-ai/fenic)

Questions? Join the Discord and ask away! For feature requests or to leave a star, visit our [GitHub](https://github.com/typedef-ai/fenic).

If this notebook helps, please give <a href="https://github.com/typedef-ai/fenic" target="_blank" rel="noopener noreferrer">fenic</a> a ⭐️ — it really helps!


## **Problem overview**

Modern AI content streams (blogs, docs, posts) are huge and messy. Teams need a fast way to:

* Clean and normalize text at scale

* Filter to on-topic items (e.g., AI/ML)

* Embed, cluster, and label articles for quick insight

* Extract technical terms and narrative intent

* Ship a lightweight “more like this” recommender without standing up infra

This notebook shows a reproducible path from raw text → structured features → clusters → exemplars → recommendations.




## **What you’ll build**

* **Data cleaning & on-topic gate:** Regex \+ optional LLM check to keep AI/ML content.

* **Embeddings at scale:** Vectorize clipped text

* **Topic clustering:** KMeans on embeddings with centroid distances and exemplars.

* **Semantic enrichment:**

  * `semantic.extract` to pull models/libraries/datasets/metrics

  * Few-shot `semantic.classify` to tag narrative intent

* **Complexity buckets:** Short/medium/long \+ code flag for triage.

* **Exports:** A tidy feature table (DuckDB \+ CSV).

* **Cluster report:** One exemplar \+ counts per cluster.

* **Recsys hooks:**

  * Top-N items closest to each centroid (cluster “profiles”)

  * Query-time “more like this” using cosine similarity


## **Who this is for**

* **Data/ML practitioners** who want a compact, production-lean pipeline for content analysis.

* **Analytics & research teams** curating large article/link corpora into themes and summaries.

* **PMs/Tech leads** prototyping discovery features (“more like this”) before building services.

## Install Required Libs and Packages

In [ ]:
# Install + Runtime sanity + Fenic Session ===
!pip -q install fenic datasets python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.2/39.2 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.1/536.1 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.6/229.6 kB 11.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

## **Step 1: Imports, Sanity Checks, and Fenic Session**

Set up the environment (libraries, credentials) and create a fresh Fenic session that we’ll reuse in later steps.

### **What this cell does**

* Reads your `OPENAI_API_KEY` from Colab’s Runtime → *Run time env vars…*.

* Creates a unique Fenic session (name includes a timestamp to avoid stale state).

* Configures **one LLM** (for classification/summaries) and **one embedding model** (for clustering).

* Runs a *local* sanity check so you can verify the session is healthy.

### **Why it matters**

* Fenic’s **semantic** features (embed, classify, extract) rely on your LLM/embedding providers. Getting the session right here prevents confusing type/quotas errors later.


In [ ]:
import os
import getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

OpenAI API Key:··········


In [ ]:
# --- Credentials -------------------------------------------------------------
# Fenic uses your OpenAI key for both LLM and embedding calls by default.
# In Colab: Runtime ▸ Run time env vars… add OPENAI_API_KEY (or set it below).
OPENAI_KEY = os.environ.get("OPENAI_API_KEY")
CLIENT = OpenAI(api_key=OPENAI_KEY) if OPENAI_KEY else None
assert OPENAI_KEY and len(OPENAI_KEY) > 10, (
    "Set OPENAI_API_KEY in Runtime ▸ Run time env vars… before running the notebook."
)

In [ ]:
# --- Imports & paths ---------------------------------------------------------
import time
import random
import csv, json

import fenic as fc
from fenic import ClassDefinition, ClassifyExample, ClassifyExampleCollection
from datasets import load_dataset # To load Medium articles dataset from HF

from pydantic import BaseModel, Field
from typing import List
from pathlib import Path

# Where we’ll drop any small artifacts (CSVs, reports) later
OUT_DIR = "/content/out"
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

LLM_MODEL = "gpt-4o-mini"
EMB_MODEL = "text-embedding-3-small"

LLM_MAX_ROWS = 300     #  hard cap for LLM classification for step 4
RANDOM_SEED  = 42

In [ ]:
# --- Session config ----------------------------------------------------------
# Unique app name helps avoid reusing stale session state in long Colab runs.
APP_NAME = f"fenic_demo_{int(time.time())}"

semantic_cfg = fc.SemanticConfig(
    language_models={
        # Small/fast LLM for labeling & summaries in later steps
        "mini": fc.OpenAILanguageModel(model_name=LLM_MODEL, rpm=2000, tpm=4_000_000),
    },
    embedding_models={
        # Vectorizer used for clustering & similarity
        "embed": fc.OpenAIEmbeddingModel(model_name=EMB_MODEL, rpm=3000, tpm=1_000_000),
    },
    default_language_model="mini",
    default_embedding_model="embed",
)

session = fc.Session.get_or_create(
    fc.SessionConfig(
        app_name=APP_NAME,
        semantic=semantic_cfg
    )
)

# Session is alive and DataFrame ops work
_ = session.create_dataframe({"check": [1, 2, 3]}).count()
print("✅ Fenic session ready")
print(" App:", APP_NAME)
print(" Default LM :", semantic_cfg.default_language_model)
print(" Default EMB:", semantic_cfg.default_embedding_model)

OUT_DIR: /content/out


INFO:fenic._backends.local.async_utils:Created new event loop on background thread
INFO:fenic._inference.model_client:Initialized client for model gpt-4o-mini with rate limit strategy UnifiedTokenRateLimitStrategy(rpm=2000, tpm=4000000)
INFO:fenic._inference.model_client:Initialized client for model text-embedding-3-small with rate limit strategy UnifiedTokenRateLimitStrategy(rpm=3000, tpm=1000000)
INFO:fenic._backends.local.manager:Session ID: de6be87e-b6c3-421f-8260-a32d2e401cae
INFO:fenic._backends.local.execution:Execution ID: 9bbeeb79-e035-48d7-b429-6db817bdb8ad


✅ Fenic session ready
 App: fenic_demo_1764166200
 Default LM : mini
 Default EMB: embed


## **Step 2: Load and Shape the Dataset**


Bring the Medium articles dataset into a clean, consistent shape `{url, title, body}`, lightly de-duplicate, and optionally downsample so later semantic steps stay fast and affordable.

### **What this cell does**

* Loads a Hugging Face dataset locally.

* Normalizes columns to `{url, title, body}` and drops empty rows.

* Removes obvious duplicates (by `url` and by `(title, body)`).

* Optionally **samples** to `SAMPLE_N` rows for a demo-friendly run time.

* Creates a **Fenic DataFrame** (`session.create_dataframe(...)`) to use in subsequent steps.

### **Why it matters**

* Clean, consistent columns avoid schema errors in downstream Fenic ops (embed, classify, extract).

* Early de-duplication and sampling **dramatically** reduce token usage in later LLM/embedding steps.

In [ ]:
# === Utils: HF dataset to Fenic DataFrame ===

def load_medium_articles(
    session,
    dataset_id: str = "fabiochiu/medium-articles",
    sample_n: int = 50_000,
    seed: int = 42,
    max_chars: int | None = None,
):
    """
    Load & shape Medium articles from HF Datasets into a Fenic DataFrame.

    Steps:
    1) load HF dataset (train split)
    2) harmonize columns -> {url, title, body}
    3) normalize to strings + strip
    4) drop empties (title/body)
    5) optional clip long bodies
    6) Python-side de-dup ({url} OR {(title, body)})
    7) reproducible sample if > sample_n
    8) return Fenic DF + stats dict
    """

    ds = load_dataset(dataset_id, split="train")

    # 2) column harmonization
    colnames = set(ds.column_names)
    rename_map = {}
    if "text" in colnames and "body" not in colnames:
        rename_map["text"] = "body"
    if "link" in colnames and "url" not in colnames:
        rename_map["link"] = "url"
    if rename_map:
        ds = ds.rename_columns(rename_map)

    required = {"url", "title", "body"}
    missing = required - set(ds.column_names)
    if missing:
        raise ValueError(
            f"Dataset missing required columns: {missing}. "
            f"Found: {sorted(ds.column_names)}"
        )

    # 3) normalize
    def _normalize(ex):
        ex["url"]   = "" if ex.get("url")   is None else str(ex["url"]).strip()
        ex["title"] = "" if ex.get("title") is None else str(ex["title"]).strip()
        ex["body"]  = "" if ex.get("body")  is None else str(ex["body"]).strip()
        return ex

    ds = ds.map(_normalize, desc="Normalizing fields")

    # 4) drop empties
    ds = ds.filter(lambda ex: len(ex["title"]) > 0 and len(ex["body"]) > 0,
                   desc="Filtering empties")

    # 5) optional clip
    if max_chars is not None and max_chars > 0:
        ds = ds.map(
            lambda ex: {**ex, "body": ex["body"][:max_chars]},
            desc=f"Clipping long bodies to {max_chars} chars"
        )

    # 6) convert + de-dup
    records = [{"url": ex["url"], "title": ex["title"], "body": ex["body"]} for ex in ds]
    seen_urls = set()
    seen_title_body = set()
    deduped = []
    for r in records:
        k1 = r["url"]
        k2 = (r["title"], r["body"])
        if k1 in seen_urls or k2 in seen_title_body:
            continue
        seen_urls.add(k1)
        seen_title_body.add(k2)
        deduped.append(r)

    # 7) reproducible sample
    if len(deduped) > sample_n:
        rng = random.Random(seed)
        deduped = rng.sample(deduped, sample_n)

    # 8) Fenic DF
    df = session.create_dataframe(deduped)

    # Stats for logging
    stats = {
        "dataset_id": dataset_id,
        "raw_rows": len(records),
        "after_filter": len(records),  # identical to raw_rows post map/filter since we rebuilt from ds
        "after_dedup": len(deduped) if len(deduped) <= sample_n else sample_n,
        "sample_n": sample_n,
        "max_chars": max_chars,
    }
    return df, stats

In [ ]:
# === Config + Dataset load (Medium Articles) ===

DATASET     = "fabiochiu/medium-articles"  # HF dataset id
SAMPLE_N    = 50_000                       # keep later steps fast/cost-aware
RANDOM_SEED = 42                            # reproducible sampling
MAX_CHARS   = None                          # e.g., 8000 to clip very long bodies

df, stats = load_medium_articles(
    session=session,
    dataset_id=DATASET,
    sample_n=SAMPLE_N,
    seed=RANDOM_SEED,
    max_chars=MAX_CHARS,
)

# Quick peek + concise log
df.show(5)
print(
    f"Loaded '{stats['dataset_id']}'. "
    f"rows(after_dedup/sample): {stats['after_dedup']:,} "
    f"| sample_n={stats['sample_n']:,} "
    f"| max_chars={stats['max_chars']}"
)

README.md: 0.00B [00:00, ?B/s]

medium_articles.csv:   0%|          | 0.00/1.04G [00:00<?, ?B/s]

medium_articles_no_text.csv:   0%|          | 0.00/49.9M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Normalizing fields:   0%|          | 0/384736 [00:00<?, ? examples/s]

Filtering empties:   0%|          | 0/384736 [00:00<?, ? examples/s]

INFO:fenic._backends.local.execution:Execution ID: 7134f4cc-5fa1-4ee2-99dd-0e67370b1cb0
INFO:fenic.api.dataframe.dataframe:Query executed in 8.99ms, returned 50,000 rows, language model cost: $0.000000, embedding model cost: $0.000000


┌────────────────────────────────┬────────────────────────────────┬────────────────────────────────┐
│ url                            ┆ title                          ┆ body                           │
╞════════════════════════════════╪════════════════════════════════╪════════════════════════════════╡
│ https://medium.com/@culturetri ┆ The Most Beautiful Hidden      ┆ The hunt for the perfect beach │
│ p/the-most-beautiful-hidden-be ┆ Beaches in the World           ┆ means that many of the best    │
│ aches-in-the-world-a1706f3a63b ┆                                ┆ spots in the world are well    │
│ f                              ┆                                ┆ known to holidaymakers,        │
│                                ┆                                ┆ leaving them overcrowded at    │
│                                ┆                                ┆ the best of times. We’ve       │
│                                ┆                                ┆ delved a little deeper 

## **Step 3: Normalize Text and Compute Simple Signals**

Create lightweight features we’ll reuse: normalized article text (`body_norm`), a boolean `has_code` flag, and length metrics (`char_len`, `title_len`).

### **What this cell does**

* Drops null bodies (prevents schema/type issues later).

* Normalizes text (lowercase \+ whitespace collapse) to stabilize token counts.

* Heuristically flags code by stripping obvious code markers and checking if length shrinks.

* Adds simple length features for later bucketing and filtering.

### **Why it matters**

* Clean, normalized text makes embedding/classification more stable and cheaper.

* `has_code` helps us segment content (e.g., tutorials vs think pieces) without using an LLM.

* Early stats help you spot odd datasets before heavy steps.

In [ ]:
# === Cleaning & simple signals (normalize text, code flag, lengths) ===

# Guard against nulls early (saves surprises downstream)
df_nonnull = df.filter(fc.col("body").is_not_null())

Create a heuristic regex for *“does this article likely contain code?”* to match Markdown fences, HTML `<code>` tags, and common Python keywords/imports.



In [ ]:
# NOTE: use \b (word boundary) — do NOT double-escape it.
CODE_RE = r"```|<code>|</code>|import [A-Za-z_]+|\bdef\b|\bclass\b"

# Build normalized text + simple features
df_clean = (
    df_nonnull
    # Lowercase + collapse whitespace -> stable for later matching/length stats
    .with_column("body_lower", fc.text.lower(fc.col("body")))
    .with_column("body_norm",  fc.text.regexp_replace(fc.col("body_lower"), r"\s+", " "))

    # "has_code": compare length before/after stripping telltale code tokens
    .with_column("body_code_stripped", fc.text.regexp_replace(fc.col("body"), CODE_RE, ""))
    .with_column("has_code", fc.text.length(fc.col("body")) > fc.text.length(fc.col("body_code_stripped")))

    # quick size signals
    .with_column("char_len",  fc.text.length(fc.col("body_norm")))
    .with_column("title_len", fc.text.length(fc.col("title")))

    # tidy up intermediates
    .drop("body_lower", "body_code_stripped")

    # Cache because downstream steps will read this multiple times
    .cache()
)

# Preview a few rows
df_clean.show(8)

INFO:fenic._backends.local.execution:Execution ID: 66e4591e-9271-4faf-bc27-480c1d0688de
INFO:fenic.api.dataframe.dataframe:Query executed in 6852.65ms, returned 50,000 rows, language model cost: $0.000000, embedding model cost: $0.000000


┌────────────────┬───────────────┬───────────────┬───────────────┬──────────┬──────────┬───────────┐
│ url            ┆ title         ┆ body          ┆ body_norm     ┆ has_code ┆ char_len ┆ title_len │
╞════════════════╪═══════════════╪═══════════════╪═══════════════╪══════════╪══════════╪═══════════╡
│ https://medium ┆ The Most      ┆ The hunt for  ┆ the hunt for  ┆ false    ┆ 5467     ┆ 46        │
│ .com/@culturet ┆ Beautiful     ┆ the perfect   ┆ the perfect   ┆          ┆          ┆           │
│ rip/the-most-b ┆ Hidden        ┆ beach means   ┆ beach means   ┆          ┆          ┆           │
│ eautiful-hidde ┆ Beaches in    ┆ that many of  ┆ that many of  ┆          ┆          ┆           │
│ n-beaches-in-t ┆ the World     ┆ the best      ┆ the best      ┆          ┆          ┆           │
│ he-world-a1706 ┆               ┆ spots in the  ┆ spots in the  ┆          ┆          ┆           │
│ f3a63bf        ┆               ┆ world are     ┆ world are     ┆          ┆          ┆   

In [ ]:
# Basic sanity stats
total_rows = df_clean.count()
with_code  = df_clean.filter(fc.col("has_code")).count()

avg_len_row = (
    df_clean.select(fc.avg(fc.col("char_len")).alias("avg_len"))
            .to_pylist()[0]
)
avg_len = avg_len_row["avg_len"] or 0

print(f"Rows: {total_rows:,}  |  has_code: {with_code:,} ({(with_code / max(total_rows, 1))*100:.1f}%)  |  avg_char_len: {int(avg_len):,}")

INFO:fenic._backends.local.execution:Execution ID: b52cec6c-aee6-4db4-b466-8161ac9affa0
INFO:fenic._backends.local.execution:Execution ID: 7e72dabc-c9df-4d7d-9673-d096f53430e5
INFO:fenic._backends.local.execution:Execution ID: 476d320e-5288-4bd1-9c38-c79cd113a18d
INFO:fenic.api.dataframe.dataframe:Query executed in 823.80ms, returned 1 rows, language model cost: $0.000000, embedding model cost: $0.000000


Rows: 50,000  |  has_code: 5,990 (12.0%)  |  avg_char_len: 5,053


## **🧭 Step 4: On-topic Filter (Regex Seed to Optional LLM Refine)**


Keep only articles that are truly about AI/ML so downstream steps (embeddings, clustering, recsys) are cheaper, faster, and higher-quality.

### What this cell does:

**(Stage 1) Lexical seed (zero-token prefilter):**
   Uses `contains_any()` against `body_norm` to keep rows that mention core AI/ML terms. This is fast, deterministic, removes obvious noise, and requires **no API keys** (which means no cost!).

**(Stage 2) LLM refine (small, capped sample):**
   If `OPENAI_API_KEY` is set, we sample up to `LLM_MAX_ROWS` from the Stage 1 hits and run `semantic.classify()` with explicit **ClassDefinitions**:

- **OnTopic:** Substantive AI/ML (architectures, training/eval, embeddings/vector search, LLM apps/agents, MLOps, data pipelines, or AI policy/ethics tied to systems).

- **OffTopic:** Everything else or only passing AI mentions.


### Knobs you can tune:

* `TOPIC_TERMS`: lexical dictionary (expand/contract to adjust recall).
* `LLM_MAX_ROWS`: cap for the semantic refine stage (balances quality vs. spend).
* `RANDOM_SEED`: keeps the refine sample reproducible.
* `has_llm_key`: automatically detected; if no key is present, we fall back to regex-only.

In [ ]:
# Simple lexical scope for AI/ML

TOPIC_TERMS = [
    "artificial intelligence", "machine learning", "deep learning",
    "neural network", "neural networks",
    "llm", "transformer", "transformers",
    "embedding", "embeddings", "vector search",
    "diffusion", "stable diffusion",
    "reinforcement learning", "rlhf", "mlops"
]

# ---- Stage 1: fast lexical seed (token-free) ----
df_seed = (
    df_clean
    .with_column("on_topic_seed", fc.col("body_norm").contains_any(TOPIC_TERMS))
    .filter(fc.col("on_topic_seed") == True)
    .drop("on_topic_seed")
)

before  = df_clean.count()
after_1 = df_seed.count()
print(f"[Stage 1] Lexical seed size: {after_1:,} / {before:,} ({(after_1/max(before,1))*100:.1f}%)")

INFO:fenic._backends.local.execution:Execution ID: a89936ac-f1d6-4b5b-8906-c5410fcccb26
INFO:fenic._backends.local.execution:Execution ID: 15940d51-3dd0-4549-bae5-9e8db5e8c0f2


[Stage 1] Lexical seed size: 3,893 / 50,000 (7.8%)


In [ ]:
# ---- Stage 2: optional LLM refine on a capped subset ----

class_defs = [
    ClassDefinition(
        label="OnTopic",
        description=(
            "The article is materially about artificial intelligence or machine learning. "
            "Examples include: model architectures (e.g., transformers, diffusion), training/eval, "
            "embeddings/vector search, LLM applications/agents, MLOps, data pipelines for ML, "
            "AI product engineering, or policy/ethics specifically tied to AI/ML systems."
        ),
    ),
    ClassDefinition(
        label="OffTopic",
        description=(
            "The article is not substantially about AI/ML. General business/marketing/lifestyle, "
            "unrelated software topics, or only a passing mention of AI without technical or substantive focus."
        ),
    ),
]

has_llm_key = bool(os.getenv("OPENAI_API_KEY"))

if has_llm_key and after_1 > 0:
    take_n  = min(after_1, LLM_MAX_ROWS)
    # Keep it deterministic/cost-bounded; .limit is fine here since upstream is deterministic
    df_take = df_seed.limit(take_n)

    df_labeled = df_take.with_column(
        "label",
        fc.semantic.classify(
            "body_norm",
            classes=class_defs,           # explicit ontology
            model_alias="mini"            # or your default from semantic_cfg if set
        )
    )
    df_on = df_labeled.filter(fc.col("label") == "OnTopic").drop("label")

    kept = df_on.count()
    print(f"[Stage 2] LLM refine kept: {kept:,} / {take_n:,} ({(kept/max(take_n,1))*100:.1f}%)")
else:
    # No key (or no rows) to lexical-only path
    df_on = df_seed

# ---- Final preview ----
df_on = df_on.cache()   # persist in-session
final_after = df_on.count()
print(f"Final on-topic rows used downstream: {final_after:,}")
df_on.show(8)

INFO:fenic._backends.local.execution:Execution ID: 077fb7a6-b061-407c-afc1-3e645dc4bf9f
INFO:fenic._inference.model_client:Creating batch adb997f4-3c37-48a4-9b11-11906678c0a3 with 300 requests for semantic.classify using (model: gpt-4o-mini)
INFO:fenic._inference.model_client:Processing batch adb997f4-3c37-48a4-9b11-11906678c0a3 with 300 requests for semantic.classify using (model: gpt-4o-mini)
Submitting requests for batch: adb997f4-3c37-48a4-9b11-11906678c0a3 (model: gpt-4o-mini): 100%|██████████| 300/300 [00:05<00:00, 56.93req/s, estimated_input_tokens=537724, estimated_output_tokens=19200]
INFO:fenic._inference.model_client:Batch adb997f4-3c37-48a4-9b11-11906678c0a3: Submitted 300 unique requests with Input Tokens: 537724, Output Tokens: 19200, Total Tokens: 556924
Awaiting responses for batch adb997f4-3c37-48a4-9b11-11906678c0a3 (model: gpt-4o-mini): 100%|██████████| 300/300 [00:01<00:00, 237.09res/s]
INFO:fenic._inference.model_client:Batch adb997f4-3c37-48a4-9b11-11906678c0a3: C

[Stage 2] LLM refine kept: 185 / 300 (61.7%)


INFO:fenic._inference.model_client:Creating batch 0c7db9b8-2e6b-4f6e-a68a-1873cba90c5c with 300 requests for semantic.classify using (model: gpt-4o-mini)
INFO:fenic._inference.model_client:Processing batch 0c7db9b8-2e6b-4f6e-a68a-1873cba90c5c with 300 requests for semantic.classify using (model: gpt-4o-mini)
Submitting requests for batch: 0c7db9b8-2e6b-4f6e-a68a-1873cba90c5c (model: gpt-4o-mini): 100%|██████████| 300/300 [00:04<00:00, 74.58req/s, estimated_input_tokens=537724, estimated_output_tokens=19200]
INFO:fenic._inference.model_client:Batch 0c7db9b8-2e6b-4f6e-a68a-1873cba90c5c: Submitted 300 unique requests with Input Tokens: 537724, Output Tokens: 19200, Total Tokens: 556924
Awaiting responses for batch 0c7db9b8-2e6b-4f6e-a68a-1873cba90c5c (model: gpt-4o-mini): 100%|██████████| 300/300 [00:09<00:00, 30.73res/s]
INFO:fenic._inference.model_client:Batch 0c7db9b8-2e6b-4f6e-a68a-1873cba90c5c: Completed with 300 responses from gpt-4o-mini
INFO:fenic._backends.local.execution:Executi

Final on-topic rows used downstream: 185
┌────────────────┬───────────────┬───────────────┬───────────────┬──────────┬──────────┬───────────┐
│ url            ┆ title         ┆ body          ┆ body_norm     ┆ has_code ┆ char_len ┆ title_len │
╞════════════════╪═══════════════╪═══════════════╪═══════════════╪══════════╪══════════╪═══════════╡
│ https://medium ┆ The Impact Of ┆ Bots are      ┆ bots are      ┆ false    ┆ 6541     ┆ 40        │
│ .com/into-adva ┆ AI Chatbots   ┆ already a     ┆ already a     ┆          ┆          ┆           │
│ nced-procureme ┆ on            ┆ reality for   ┆ reality for   ┆          ┆          ┆           │
│ nt/procurement ┆ Procurement   ┆ some          ┆ some          ┆          ┆          ┆           │
│ -and-ai-chatbo ┆               ┆ Procurement   ┆ procurement   ┆          ┆          ┆           │
│ ts-bots-are-al ┆               ┆ organizations ┆ organizations ┆          ┆          ┆           │
│ ready-a-realit ┆               ┆ . Indeed,     ┆

## **Step 5: Clip Text to Embed in Fenic**

This step turns each article into an **embedding vector** using Fenic’s built-in semantic APIs. We first **clip** the normalized body (`body_norm`) to a fixed length (for speed/cost), then compute embeddings **inside Fenic**.

### **What this cell does**

1. **Guard & prepare**

   * Ensures `body_norm` isn’t null.
   * Adds `char_len` and creates `body_clip` by taking the **first `CLIP_N` characters** (regex is used so newlines are handled).
   * Adds `clip_len` for quick sanity checks.

2. **Inspect & sanity-check**

   * Shows a tiny preview (`clip_preview`) of each clipped body.
   * Prints stats: total rows prepared, how many were over the limit (and thus clipped), and the maximum `clip_len` seen.

3. **Embed all rows**

   * Calls `fc.semantic.embed("body_clip")` with the **default embedding model**.
   * Produces `df_emb` with: `url`, `title`, `body_clip`, `char_len`, `clip_len`, `emb` (Fenic **EmbeddingType**).


### **Knobs you can tune**

* `CLIP_N = 1500` (default here).

  * Increase for richer semantics (more tokens), decrease for speed/cost.
* *(Optional)* `EMBED_ROW_CAP = None`

  * If you expect very large runs in Colab, define `EMBED_ROW_CAP = 10_000` (for example) and insert a single `.limit(EMBED_ROW_CAP)` just **before** `.with_column("emb", ...)`.

In [ ]:
CLIP_N = 1500

print(f"[Embeddings] Preparing rows for embedding | char_clip={CLIP_N}")

# 0) Prep: filter, length, clip
df_prepared = (
    df_on
    .filter(fc.col("body_norm").is_not_null())
    .with_column("char_len", fc.text.length("body_norm"))
    # Keep first ≤ CLIP_N chars; (?s) makes dot match newlines
    .with_column(
        "body_clip",
        fc.text.regexp_replace("body_norm", rf"(?s)^(.{{0,{CLIP_N}}}).*", "$1")
    )
    .with_column("clip_len", fc.text.length("body_clip"))
)

[Embeddings] Preparing rows for embedding | char_clip=1500


In [ ]:
# Quick peek (optional)
df_prepared.select(
    "title",
    "char_len",
    "clip_len",
    fc.text.regexp_replace("body_clip", r"(?s)^(.{0,30}).*", "$1").alias("clip_preview"),
).show(6)

# Stats
total_rows = df_prepared.count()
clipped_rows = df_prepared.filter(fc.col("char_len") > CLIP_N).count()
_max_row = (
    df_prepared.sort("clip_len", ascending=False)
               .limit(1)
               .select("clip_len")
               .to_pylist()
)
max_clip_len = _max_row[0]["clip_len"] if _max_row else 0
print(f"Rows prepared: {total_rows:,} | originally-over-limit clipped: {clipped_rows:,} | max clip_len observed: {max_clip_len}")

# 1) Embed *all* prepared rows using the default embedding model and cache it to avoid recomputing
df_emb = (
    df_prepared
    .with_column("emb", fc.semantic.embed("body_clip"))
    .select("url", "title", "body_clip", "char_len", "clip_len", "emb")
    .cache()
)

# Peek embedded rows
df_emb.select(
    "title", "char_len", "clip_len",
    fc.text.regexp_replace("body_clip", r"(?s)^(.{0,30}).*", "$1").alias("clip_preview"),
).show(6)

INFO:fenic._backends.local.execution:Execution ID: 51fa1979-0502-4d80-bfdf-20d7211abaa6
INFO:fenic.api.dataframe.dataframe:Query executed in 199.55ms, returned 185 rows, language model cost: $0.000000, embedding model cost: $0.000000
INFO:fenic._backends.local.execution:Execution ID: 05b66433-1762-4e22-af15-3f526576f92f
INFO:fenic._backends.local.execution:Execution ID: 3fad1d8a-a210-4d7a-b187-479b9af6a33e


┌───────────────────────────────────────────┬──────────┬──────────┬────────────────────────────────┐
│ title                                     ┆ char_len ┆ clip_len ┆ clip_preview                   │
╞═══════════════════════════════════════════╪══════════╪══════════╪════════════════════════════════╡
│ The Impact Of AI Chatbots on Procurement  ┆ 6541     ┆ 1500     ┆ bots are already a reality for │
│ Improving Sales Conversion Rates at       ┆ 6917     ┆ 1500     ┆ by rich pacheco, william dunn, │
│ Credit Risk Monitor®                      ┆          ┆          ┆                                │
│ Black Friday 2021 Merchandising           ┆ 11957    ┆ 1500     ┆ 11 online merchandising tactic │
│ …                                         ┆ …        ┆ …        ┆ …                              │
│ Product Recommender using Amazon Review   ┆ 13679    ┆ 1500     ┆ what is the problem? why do we │
│ dataset                                   ┆          ┆          ┆                        

INFO:fenic._backends.local.execution:Execution ID: 58cb2c5b-e0a2-4b29-b56b-42f3e86eef90
INFO:fenic.api.dataframe.dataframe:Query executed in 116.22ms, returned 1 rows, language model cost: $0.000000, embedding model cost: $0.000000
INFO:fenic._backends.local.execution:Execution ID: f196732b-2030-4dac-ab33-6ad100dabf92
INFO:fenic._inference.model_client:Creating batch a12c6cf3-2b0f-47bf-9380-3136fca8bb40 with 185 requests for semantic.embed using (model: text-embedding-3-small)
INFO:fenic._inference.model_client:Processing batch a12c6cf3-2b0f-47bf-9380-3136fca8bb40 with 185 requests for semantic.embed using (model: text-embedding-3-small)


Rows prepared: 185 | originally-over-limit clipped: 172 | max clip_len observed: 1500


Submitting requests for batch: a12c6cf3-2b0f-47bf-9380-3136fca8bb40 (model: text-embedding-3-small): 100%|██████████| 185/185 [00:01<00:00, 99.93req/s, estimated_input_tokens=54256, estimated_output_tokens=0] 
INFO:fenic._inference.model_client:Batch a12c6cf3-2b0f-47bf-9380-3136fca8bb40: Submitted 185 unique requests with Input Tokens: 54256, Output Tokens: 0, Total Tokens: 54256
Awaiting responses for batch a12c6cf3-2b0f-47bf-9380-3136fca8bb40 (model: text-embedding-3-small): 100%|██████████| 185/185 [00:03<00:00, 47.31res/s]
INFO:fenic._inference.model_client:Batch a12c6cf3-2b0f-47bf-9380-3136fca8bb40: Completed with 185 responses from text-embedding-3-small
INFO:fenic.api.dataframe.dataframe:Query executed in 5916.86ms, returned 185 rows, language model cost: $0.000000, embedding model cost: $0.001085


┌───────────────────────────────────────────┬──────────┬──────────┬────────────────────────────────┐
│ title                                     ┆ char_len ┆ clip_len ┆ clip_preview                   │
╞═══════════════════════════════════════════╪══════════╪══════════╪════════════════════════════════╡
│ The Impact Of AI Chatbots on Procurement  ┆ 6541     ┆ 1500     ┆ bots are already a reality for │
│ Improving Sales Conversion Rates at       ┆ 6917     ┆ 1500     ┆ by rich pacheco, william dunn, │
│ Credit Risk Monitor®                      ┆          ┆          ┆                                │
│ Black Friday 2021 Merchandising           ┆ 11957    ┆ 1500     ┆ 11 online merchandising tactic │
│ …                                         ┆ …        ┆ …        ┆ …                              │
│ Product Recommender using Amazon Review   ┆ 13679    ┆ 1500     ┆ what is the problem? why do we │
│ dataset                                   ┆          ┆          ┆                        

## **Step 6: K-Means Clustering on Embeddings**

We’ve now got per-article embeddings (Step 5). K-Means groups semantically similar articles into *clusters*, so we can:

* see dominant themes,

* pick a representative example per theme,

* and summarize or label each theme downstream.

### **What this cell does**

1. **Sanity checks**: Ensures `df_emb` is populated and has an `emb` column of `EmbeddingType`.

2. **Clustering**: Runs `with_cluster_labels("emb", …)` to assign each row a `cluster` and also returns the cluster `centroid`.

3. **Sizes**: Counts rows per cluster to understand the distribution.

4. **Exemplars**: Computes cosine distance to the centroid and picks the closest title per cluster.

5. **Labels**: Uses the exemplar title as a simple human-readable `cluster_label`.

In [ ]:
# === Fenic-native KMeans on EmbeddingType ===

K = 10

# Require embeddings + a couple of human-readable columns
cols = set(df_emb.schema.column_names())
assert {"emb", "title"} <= cols, "Step 6 needs df_emb with 'emb' (EmbeddingType) and 'title'."

print(f"[KMeans] rows={df_emb.count()} | k={K}")

# 1) Cluster with Fenic’s semantic API
df_clustered = df_emb.semantic.with_cluster_labels(
    "emb",
    num_clusters=K,
    num_init=8,
    max_iter=100,
    label_column="cluster",
    centroid_column="centroid"
)

# 2) Peek: titles per cluster
df_clustered.select("cluster", "title").sort("cluster").show(20)

INFO:fenic._backends.local.execution:Execution ID: 68dc84c4-ed3b-4634-8891-5b5cc12bd640
INFO:fenic._backends.local.execution:Execution ID: e076c564-ab81-4502-81a9-8a74e7f9fe8d


[KMeans] rows=185 | k=10


INFO:fenic.api.dataframe.dataframe:Query executed in 284.09ms, returned 185 rows, language model cost: $0.000000, embedding model cost: $0.000000


┌─────────┬────────────────────────────────────────────────────────────────────────────────────────┐
│ cluster ┆ title                                                                                  │
╞═════════╪════════════════════════════════════════════════════════════════════════════════════════╡
│ 0       ┆ Zero to Hero: Machine Learning Competition                                             │
│ 0       ┆ “Stock Market Anomalies” and “Stock Market Anomaly Detection” Are Two Different Things │
│ 0       ┆ Sentiment Analysis of Twitter’s US Airlines Data using KNN Classification              │
│ 0       ┆ 9 Guidelines to master Scikit-learn without giving up in the middle                    │
│ 0       ┆ HDSC Stage F OSP- Restaurant Revenue Prediction                                        │
│ 0       ┆ Closed-form and Gradient Descent Regression Explained with Python                      │
│ 0       ┆ Ensemble Learning Relation With Bias and variance                              

In [ ]:
# 3) Cluster sizes
(
    df_clustered
    .group_by("cluster")
    .agg(fc.count("*").alias("count"))
    .sort(fc.col("count").desc())
    .show(K)
)

# 4) Exemplars (closest to centroid; stable ties by title)
with_scores = df_clustered.select(
    "*",
    (fc.lit(1.0) - fc.embedding.compute_similarity(
        fc.col("emb"), fc.col("centroid"), metric="cosine"
    )).alias("dist_to_centroid")
)

exemplars = (
    with_scores
    .sort(["cluster", "dist_to_centroid", "title"], ascending=[True, True, True])
    .group_by("cluster")
    .agg(
        fc.first(fc.col("title")).alias("title"),
        fc.first(fc.col("dist_to_centroid")).alias("dist_to_centroid")
    )
    .sort("cluster")
)

print("\n[Exemplars] One representative title per cluster:")
exemplars.show(K)

INFO:fenic._backends.local.execution:Execution ID: d1e418aa-e796-44b2-ba18-94eaeef7dff3
INFO:fenic.api.dataframe.dataframe:Query executed in 203.21ms, returned 10 rows, language model cost: $0.000000, embedding model cost: $0.000000
INFO:fenic._backends.local.execution:Execution ID: 7276d891-bd48-448d-a014-526c6e0fe258


┌─────────┬───────┐
│ cluster ┆ count │
╞═════════╪═══════╡
│ 3       ┆ 36    │
│ 9       ┆ 24    │
│ 7       ┆ 21    │
│ 8       ┆ 20    │
│ 4       ┆ 19    │
│ 0       ┆ 19    │
│ 5       ┆ 16    │
│ 1       ┆ 15    │
│ 2       ┆ 13    │
│ 6       ┆ 2     │
└─────────┴───────┘

[Exemplars] One representative title per cluster:


INFO:fenic.api.dataframe.dataframe:Query executed in 208.63ms, returned 10 rows, language model cost: $0.000000, embedding model cost: $0.000000


┌─────────┬─────────────────────────────────────────────────────────────────────┬──────────────────┐
│ cluster ┆ title                                                               ┆ dist_to_centroid │
╞═════════╪═════════════════════════════════════════════════════════════════════╪══════════════════╡
│ 0       ┆ Student’s marks prediction using python                             ┆ 0.25858          │
│ 1       ┆ Innovation in Regulated Industries is a Key to Social Impact        ┆ 0.269786         │
│ 2       ┆ After liner regression, logistic regression, Neural network What is ┆ 0.239573         │
│         ┆ next?                                                               ┆                  │
│ 3       ┆ Are you Leveraging AI to Train your Workforce?                      ┆ 0.267519         │
│ 4       ┆ Fundamentals of ConvNets                                            ┆ 0.24554          │
│ 5       ┆ Connecting the Dots (Python, Spark, and Kafka)                      ┆ 0.25946  

In [ ]:
# 5) Lightweight labels from exemplar titles
labels = exemplars.select("cluster", fc.col("title").alias("cluster_label"))
df_labeled = df_clustered.join(labels, on="cluster", how="left")

# Pretty preview
df_labeled.select("cluster", "cluster_label", "title").sort("cluster").show(20)

print("✅ KMeans clustering complete.")

INFO:fenic._backends.local.execution:Execution ID: 2bef9247-2bcd-4810-9440-577c9b2ee0d9
INFO:fenic.api.dataframe.dataframe:Query executed in 352.07ms, returned 185 rows, language model cost: $0.000000, embedding model cost: $0.000000


┌─────────┬────────────────────────────────────────────┬───────────────────────────────────────────┐
│ cluster ┆ cluster_label                              ┆ title                                     │
╞═════════╪════════════════════════════════════════════╪═══════════════════════════════════════════╡
│ 0       ┆ Student’s marks prediction using python    ┆ Zero to Hero: Machine Learning            │
│         ┆                                            ┆ Competition                               │
│ 0       ┆ Student’s marks prediction using python    ┆ “Stock Market Anomalies” and “Stock       │
│         ┆                                            ┆ Market Anomaly Detection” Are Two         │
│         ┆                                            ┆ Different Things                          │
│ 0       ┆ Student’s marks prediction using python    ┆ Sentiment Analysis of Twitter’s US        │
│         ┆                                            ┆ Airlines Data using KNN Classifica

## **Step 7: Technical Terms via `semantic.extract`**


### **What this cell does**

* Defines a lightweight Pydantic schema (`TechTerms`) for the fields you care about (models, libraries, datasets, metrics).

* Uses **Fenic’s** `semantic.extract(text_col, PydanticModel)` to run an LLM over each row’s `body_clip` and return a **struct** column (`tech`) that matches your schema.

* Shows the results by selecting fields from the struct and (optionally) exploding one list (models) to a long form view.

### **Why it matters**

* You get **structured fields** out of unstructured articles with a single op — perfect for later clustering, faceting, or building simple dashboards (“show me clusters mentioning `Transformers` \+ `COCO`”).

* The schema keeps your output predictable and robust for teaching: empty buckets are just `[]`, not `null`.


In [ ]:
# Assumes df_emb exists from Step 5 and includes: title, body_clip, emb, etc.

# ---- Safety checks -------------------------------------------------------
required_cols = {"title", "body_clip"}
missing = required_cols - set(df_emb.schema.column_names())
assert not missing, f"Step 7 needs columns {sorted(required_cols)} in df_emb; missing: {sorted(missing)}"

# Use the full dataset for simplicity
df_src = df_emb

In [ ]:
# ---- Define the structured schema we want from the LLM -------------------
class TechTerms(BaseModel):
    models:    List[str] = Field(default_factory=list, description="ML/DL models or architectures (e.g., BERT, ResNet, LSTM).")
    libraries: List[str] = Field(default_factory=list, description="Libraries/frameworks (e.g., PyTorch, TensorFlow, scikit-learn).")
    datasets:  List[str] = Field(default_factory=list, description="Public datasets/corpora (e.g., ImageNet, COCO, GLUE).")
    metrics:   List[str] = Field(default_factory=list, description="Evaluation metrics (e.g., F1, BLEU, RMSE, accuracy).")

print("[Extract] Mining technical terms from clipped article text…")

[Extract] Mining technical terms from clipped article text…


In [ ]:
# ---- Run extraction: returns a struct column 'tech' matching TechTerms ---
df_terms = df_src.select(
    "*",
    fc.semantic.extract(
        fc.col("body_clip"),
        TechTerms
    ).alias("tech")
)

In [ ]:
# ---- Preview: show lists per row -----------------------------------------
df_terms.select(
    "title",
    fc.col("tech")["models"].alias("models"),
    fc.col("tech")["libraries"].alias("libraries"),
    fc.col("tech")["datasets"].alias("datasets"),
    fc.col("tech")["metrics"].alias("metrics"),
).show(10)

INFO:fenic._backends.local.execution:Execution ID: 2b0677ac-4820-4fac-a6fd-391ca7d47779
INFO:fenic._inference.model_client:Creating batch 1cd36121-502d-4300-970e-6ba9723fb8e5 with 185 requests for semantic.extract using (model: gpt-4o-mini)
INFO:fenic._inference.model_client:Processing batch 1cd36121-502d-4300-970e-6ba9723fb8e5 with 185 requests for semantic.extract using (model: gpt-4o-mini)
Submitting requests for batch: 1cd36121-502d-4300-970e-6ba9723fb8e5 (model: gpt-4o-mini): 100%|██████████| 185/185 [00:02<00:00, 78.16req/s, estimated_input_tokens=110961, estimated_output_tokens=189440]
INFO:fenic._inference.model_client:Batch 1cd36121-502d-4300-970e-6ba9723fb8e5: Submitted 185 unique requests with Input Tokens: 110961, Output Tokens: 189440, Total Tokens: 300401
Awaiting responses for batch 1cd36121-502d-4300-970e-6ba9723fb8e5 (model: gpt-4o-mini): 100%|██████████| 185/185 [00:01<00:00, 95.85res/s]
INFO:fenic._inference.model_client:Batch 1cd36121-502d-4300-970e-6ba9723fb8e5: Co

┌─────────────────────┬─────────────────────┬────────────────┬───────────────┬─────────────────────┐
│ title               ┆ models              ┆ libraries      ┆ datasets      ┆ metrics             │
╞═════════════════════╪═════════════════════╪════════════════╪═══════════════╪═════════════════════╡
│ The Impact Of AI    ┆ ["NLP"]             ┆ []             ┆ []            ┆ []                  │
│ Chatbots on         ┆                     ┆                ┆               ┆                     │
│ Procurement         ┆                     ┆                ┆               ┆                     │
│ Improving Sales     ┆ []                  ┆ []             ┆ []            ┆ ["conversion rate"] │
│ Conversion Rates at ┆                     ┆                ┆               ┆                     │
│ Credit Risk         ┆                     ┆                ┆               ┆                     │
│ Monitor®            ┆                     ┆                ┆               ┆             

In [ ]:
# ---- 7.4 Optional: explode one bucket (models) for a long-form sample --------
models_long = (
    df_terms
    .with_column("models", fc.col("tech")["models"])  # lift list out of struct
    .explode("models")
    .filter(fc.col("models").is_not_null())
    .select("title", "models")
)

print("\n[Extract] Sample extracted models:")
models_long.show(10)

print("✅ Technical term extraction complete.")

INFO:fenic._backends.local.execution:Execution ID: db9bb42a-c52e-410b-964d-62981f4783ff
INFO:fenic._inference.model_client:Creating batch b9907089-54bf-404b-9e9a-d62d37f90166 with 185 requests for semantic.extract using (model: gpt-4o-mini)
INFO:fenic._inference.model_client:Processing batch b9907089-54bf-404b-9e9a-d62d37f90166 with 185 requests for semantic.extract using (model: gpt-4o-mini)



[Extract] Sample extracted models:


Submitting requests for batch: b9907089-54bf-404b-9e9a-d62d37f90166 (model: gpt-4o-mini): 100%|██████████| 185/185 [00:02<00:00, 71.92req/s, estimated_input_tokens=110961, estimated_output_tokens=189440]
INFO:fenic._inference.model_client:Batch b9907089-54bf-404b-9e9a-d62d37f90166: Submitted 185 unique requests with Input Tokens: 110961, Output Tokens: 189440, Total Tokens: 300401
Awaiting responses for batch b9907089-54bf-404b-9e9a-d62d37f90166 (model: gpt-4o-mini): 100%|██████████| 185/185 [00:02<00:00, 90.28res/s]
INFO:fenic._inference.model_client:Batch b9907089-54bf-404b-9e9a-d62d37f90166: Completed with 185 responses from gpt-4o-mini
INFO:fenic.api.dataframe.dataframe:Query executed in 4658.25ms, returned 111 rows, language model cost: $0.023777, embedding model cost: $0.000000


┌───────────────────────────────────────────────────────────┬──────────────────────────────────────┐
│ title                                                     ┆ models                               │
╞═══════════════════════════════════════════════════════════╪══════════════════════════════════════╡
│ Building a Simple Neural Network from Scratch             ┆ neural network                       │
│ Weekly Pentina Prompt: It’s Artificial                    ┆ generative adversarial network (GAN) │
│ The Applications and Benefits of a PreTrained Model ––    ┆ CNN                                  │
│ Kaggle’s DogsVSCats                                       ┆                                      │
│ How will tech define the post-COVID era in healthcare?    ┆ artificial intelligence              │
│ My top 5 tips to getting the best out of your analytics   ┆ machine learning                     │
│ …                                                         ┆ …                            

## **Step 8: Narrative Intent (Few-shot Classification)**

We want to understand *how* each article is written (news vs tutorial vs explainer, etc.). This helps with downstream UX (e.g., filter to tutorials) and improves summarization prompts (different tone per intent).

### **What the cell does**

* Defines a small, mutually exclusive set of labels via `ClassDefinition`.

* Provides 5 short, unambiguous few-shot examples so the model learns the boundaries.

* Calls `fc.semantic.classify("body_clip", INTENT_CLASSES, examples=examples, model_alias="mini")` to predict one label per row.

* Prints a quick sample and a label distribution to sanity-check balance.

### **Why it matters**

* Few-shot guidance improves accuracy vs zero-shot.

* Using `body_clip` (from Step 5\) keeps token usage low and speeds up runs.

* `CLASSIFY_MAX_ROWS` caps cost/time for classroom/demo settings.

In [ ]:
# Assumes df_emb exists from Step 5 and includes a short, token-friendly 'body_clip' column.

print("[Classify] Labeling narrative intent with few-shot EXAMPLES sourced from corpus…")

# Keep things snappy/cost-friendly in Colab. Bump if you want more coverage.
CLASSIFY_MAX_ROWS = 300
MODEL_ALIAS = semantic_cfg.default_language_model  # uses the "mini" alias from Step 1

# Guard rails: required inputs for this step
required_cols = {"title", "body_clip"}
missing = required_cols - set(df_emb.schema.column_names())
assert not missing, f"Step 8 needs columns {sorted(required_cols)} in df_emb; missing: {sorted(missing)}"

df_src = df_emb.limit(CLASSIFY_MAX_ROWS)

[Classify] Labeling narrative intent with few-shot EXAMPLES sourced from corpus…


In [ ]:
# ---- Class set (use ClassDefinition for clearer intent boundaries) -------
INTENT_CLASSES = [
    ClassDefinition(label="news/announcement",
                    description="Objective report about a new event, release, partnership, funding, or policy."),
    ClassDefinition(label="tutorial/how-to",
                    description="Step-by-step instructions, hands-on walkthroughs, code or commands."),
    ClassDefinition(label="opinion/thinkpiece",
                    description="Subjective commentary, essays, or argumentative takes."),
    ClassDefinition(label="research/explainer",
                    description="Concept explanations, literature overviews, or technical deep dives."),
    ClassDefinition(label="case-study/showcase",
                    description="Real-world application or project write-up; results and lessons."),
]

Helper: regex match via [`text.regexp_replace`](https://docs.fenic.ai/latest/reference/fenic/api/functions/text/?h=regexp_replace#fenic.api.functions.text.regexp_replace) + length delta

**NOTE**: `(?i)` makes the regex case-insensitive in many engines; works here via regexp.

In [ ]:
# pattern should include (?i) if you want case-insensitive matching.
def pick(label: str, pattern: str):
    t = fc.col("title")
    replaced = fc.text.regexp_replace(t, pattern, "")
    matched = fc.text.length(replaced) < fc.text.length(t)
    return (
        df_src
        .filter(matched)
        .select(
            fc.col("body_clip").alias("input"),
            fc.lit(label).alias("output"),
        )
        .limit(1)  # one crisp example per label
    )

Build a tiny, crisp, non-overlapping example set from your own rows.

In [ ]:
ex_news      = pick("news/announcement",   r"(?i)(announce|launch|series [abc]|funding)")
ex_tutorial  = pick("tutorial/how-to",     r"(?i)(how to|tutorial|step[- ]by[- ]step|guide)")
ex_opinion   = pick("opinion/thinkpiece",  r"(?i)(^why\s|opinion|my take|thoughts)")
ex_explainer = pick("research/explainer",  r"(?i)(explain|explainer|what is|overview of|mechanism)")
ex_case      = pick("case-study/showcase", r"(?i)(case study|our rollout|architecture|postmortem|we cut|reduced)")

# Union and de-dupe examples
df_examples = (
    ex_news
    .union(ex_tutorial)
    .union(ex_opinion)
    .union(ex_explainer)
    .union(ex_case)
    .drop_duplicates()
)

Convert Fenic to Polars and build a [`ClassifyExampleCollection`](https://docs.fenic.ai/latest/reference/fenic/?h=classifyexamplecollection#fenic.ClassifyExampleCollection)

In [ ]:
examples_pl = df_examples.collect("polars").data
examples = ClassifyExampleCollection.from_polars(examples_pl)

INFO:fenic._backends.local.execution:Execution ID: 8480b821-7854-4e32-b41f-6085544b16e2
INFO:fenic.api.dataframe.dataframe:Query executed in 38.38ms, returned 4 rows, language model cost: $0.000000, embedding model cost: $0.000000


In [ ]:
# Your INTENT_CLASSES should already be defined earlier.
df_intent = df_src.select(
    "*",
    fc.semantic.classify(
        "body_clip",
        INTENT_CLASSES,
        examples=examples,
        model_alias=MODEL_ALIAS,
        temperature=0
    ).alias("intent")
).cache()

df_intent.select("title", "intent").show(15)

INFO:fenic._backends.local.execution:Execution ID: 834e7fd7-cebf-4c3b-9f7f-fffe154c115a
INFO:fenic._inference.model_client:Creating batch 28a1c1ae-b31e-44aa-8cb6-bd3a80f06352 with 185 requests for semantic.classify using (model: gpt-4o-mini)
INFO:fenic._inference.model_client:Processing batch 28a1c1ae-b31e-44aa-8cb6-bd3a80f06352 with 185 requests for semantic.classify using (model: gpt-4o-mini)
Submitting requests for batch: 28a1c1ae-b31e-44aa-8cb6-bd3a80f06352 (model: gpt-4o-mini): 100%|██████████| 185/185 [00:04<00:00, 37.80req/s, estimated_input_tokens=302251, estimated_output_tokens=11840]
INFO:fenic._inference.model_client:Batch 28a1c1ae-b31e-44aa-8cb6-bd3a80f06352: Submitted 185 unique requests with Input Tokens: 302251, Output Tokens: 11840, Total Tokens: 314091
Awaiting responses for batch 28a1c1ae-b31e-44aa-8cb6-bd3a80f06352 (model: gpt-4o-mini): 100%|██████████| 185/185 [00:04<00:00, 39.79res/s]
INFO:fenic._inference.model_client:Batch 28a1c1ae-b31e-44aa-8cb6-bd3a80f06352: Co

┌───────────────────────────────────────────────────────────────────────┬─────────────────────┐
│ title                                                                 ┆ intent              │
╞═══════════════════════════════════════════════════════════════════════╪═════════════════════╡
│ The Impact Of AI Chatbots on Procurement                              ┆ news/announcement   │
│ Improving Sales Conversion Rates at Credit Risk Monitor®              ┆ case-study/showcase │
│ Black Friday 2021 Merchandising                                       ┆ tutorial/how-to     │
│ Trying to Understand Tries                                            ┆ research/explainer  │
│ Building a Simple Neural Network from Scratch                         ┆ tutorial/how-to     │
│ The Human Understanding Path to Success                               ┆ opinion/thinkpiece  │
│ Zero to Hero: Machine Learning Competition                            ┆ tutorial/how-to     │
│ The State Of AI                       

In [ ]:
# ---- Quick previews of the intent distribution --------------

print("\n[Classify] Distribution:")
(
    df_intent
    .group_by("intent")
    .agg(fc.count("*").alias("count"))
    .sort("count", ascending=False)
    .show(10)
)

# Keep df_intent for downstream steps (complexity buckets, etc.)

INFO:fenic._backends.local.execution:Execution ID: 65fd97d4-ce07-44e6-a307-b33cd22e37a9
INFO:fenic.api.dataframe.dataframe:Query executed in 7.80ms, returned 5 rows, language model cost: $0.000000, embedding model cost: $0.000000



[Classify] Distribution:
┌─────────────────────┬───────┐
│ intent              ┆ count │
╞═════════════════════╪═══════╡
│ tutorial/how-to     ┆ 56    │
│ research/explainer  ┆ 54    │
│ news/announcement   ┆ 26    │
│ case-study/showcase ┆ 25    │
│ opinion/thinkpiece  ┆ 24    │
└─────────────────────┴───────┘


## **Step 9: Complexity Buckets \+ Code Flag**

### **What this cell does**

* Adds two lightweight features for each article:

  * **`complexity_bucket`**: a coarse length class (`short` / `medium` / `long`) computed from `char_len`.

  * **`has_code`**: a boolean flag that heuristically detects **inline code** or **code blocks** in the text.

* Prints quick previews:

  * A few labeled rows (`title`, `complexity_bucket`, `has_code`, `intent`)

  * Bucket counts

  * A cross-tab of bucket × code flag

### **Why it matters**

* **Downstream controls**: lets you target different processing strategies—e.g., skip LLM summarization for *long, code-heavy* docs; or prefer lighter models for *short* items.

* **User-facing filters**: handy facets for UIs (“show tutorials with code under 1k chars”).

* **Cost hygiene**: knowing length distribution helps keep token usage predictable later.


In [ ]:
# Assumes df_intent exists from Step 8 and includes: title, body_clip, char_len, intent.

# ---- Guard rails: make sure inputs exist ---------------------------------
required_cols = {"title", "body_clip", "char_len", "intent"}
missing = required_cols - set(df_intent.schema.column_names())
assert not missing, f"Step 9 needs columns {sorted(required_cols)}; missing: {sorted(missing)}"

print("[Complexity] Bucketing by length and surfacing the code flag…")

[Complexity] Bucketing by length and surfacing the code flag…


Use the same idea as Step 3, but run on `'body_clip'` (we didn't carry the original flag forward)

In [ ]:
# ----'has_code' from the clipped text (token-free; robust) ---------------

CODE_RE = r"```|<code>|</code>|import [A-Za-z_]+|\bdef\b|\bclass\b"

df_complex = (
    df_intent
    .with_column("body_code_stripped", fc.text.regexp_replace("body_clip", CODE_RE, ""))
    .with_column("has_code", fc.text.length("body_clip") > fc.text.length("body_code_stripped"))
    .drop("body_code_stripped")
)

In [ ]:
# ---- Complexity buckets from char count (short/medium/long) --------------
# Demo thresholds:
#   short  : < 1,000 chars
#   medium : 1,000–2,999
#   long   : ≥ 3,000

bucket_expr = (
    fc.when(fc.col("char_len") >= fc.lit(3000), fc.lit("long"))
      .when(fc.col("char_len") >= fc.lit(1000), fc.lit("medium"))
      .otherwise(fc.lit("short"))
)
df_complex = df_complex.with_column("complexity_bucket", bucket_expr)

In [ ]:
# ---- Quick previews ----------

print("\n[Examples] A few rows with bucket, code flag, and intent:")
df_complex.select("title", "complexity_bucket", "has_code", "intent").show(15)

print("\n[Counts] Bucket distribution:")
(
    df_complex
    .group_by("complexity_bucket")
    .agg(fc.count("*").alias("count"))
    .sort(fc.col("count").desc())
    .show(10)
)

print("\n[Cross-tab] Bucket × has_code:")
(
    df_complex
    .group_by("complexity_bucket", "has_code")
    .agg(fc.count("*").alias("count"))
    .sort(["complexity_bucket", "has_code"], ascending=[True, True])
    .show(20)
)

# Keep df_complex for the final feature table (next step).

INFO:fenic._backends.local.execution:Execution ID: 6cccd9ef-3830-47ea-a437-5bf6ed73b786
INFO:fenic.api.dataframe.dataframe:Query executed in 18.06ms, returned 185 rows, language model cost: $0.000000, embedding model cost: $0.000000
INFO:fenic._backends.local.execution:Execution ID: defcf689-763e-4c14-bbd6-e5c3768a4e5c
INFO:fenic.api.dataframe.dataframe:Query executed in 8.84ms, returned 3 rows, language model cost: $0.000000, embedding model cost: $0.000000
INFO:fenic._backends.local.execution:Execution ID: 1bd11205-c1f4-4eaf-8fdd-bb3ed2faf2b6
INFO:fenic.api.dataframe.dataframe:Query executed in 8.10ms, returned 5 rows, language model cost: $0.000000, embedding model cost: $0.000000



[Examples] A few rows with bucket, code flag, and intent:
┌─────────────────────────────────────────────┬───────────────────┬──────────┬─────────────────────┐
│ title                                       ┆ complexity_bucket ┆ has_code ┆ intent              │
╞═════════════════════════════════════════════╪═══════════════════╪══════════╪═════════════════════╡
│ The Impact Of AI Chatbots on Procurement    ┆ long              ┆ false    ┆ news/announcement   │
│ Improving Sales Conversion Rates at Credit  ┆ long              ┆ false    ┆ case-study/showcase │
│ Risk Monitor®                               ┆                   ┆          ┆                     │
│ Black Friday 2021 Merchandising             ┆ long              ┆ false    ┆ tutorial/how-to     │
│ Trying to Understand Tries                  ┆ long              ┆ false    ┆ research/explainer  │
│ Building a Simple Neural Network from       ┆ long              ┆ false    ┆ tutorial/how-to     │
│ Scratch                       

## **Step 10: Final Feature Table**

### **What this cell does**

* **Assembles one tidy table** from previous steps (clean text, lexical features, intent label, complexity bucket, and, optionally, embeddings).

* Persists the table with a **CSV** for lightweight sharing (usually without `emb` to keep file size reasonable) and **parquet** file.

### **Why it matters**

* You’ve computed several features across steps; this step **materializes a single source of truth** for downstream work (viz, BI tools, recsys protos, model training).

In [ ]:
import duckdb

print("[Export] Building final feature table...")

# ---- Pick source DF (Step 9 preferred) ----

assert df_complex is not None, "No feature found in df_complex. Run through Step 9 first."

# ---- Select/base columns (defensive: select only those that exist) ----
BASE_COLS = [
    "url", "title", "body_clip", "char_len", "clip_len",
    "has_code", "complexity_bucket", "intent",
]
have_cols = [c for c in BASE_COLS if c in df_complex.schema.column_names()]
final_df = df_complex.select(*have_cols)

[Export] Building final feature table


In [ ]:
# ---- Optional: enrich with clusters from Step 6 (join on url) ----

# Keep a friendly column order if cluster fields are present
ORDERED = [c for c in BASE_COLS + ["cluster", "cluster_label"] if c in final_df.schema.column_names()]
final_df = final_df.select(*ORDERED)

Write Parquet (preferred) and CSV directly from Fenic:

In [ ]:
final_df.write.parquet(f"{OUT_DIR}/features.parquet")
final_df.write.csv(f"{OUT_DIR}/features.csv")

print(f"✅ Wrote Parquet → {OUT_DIR}/features.parquet")
print(f"✅ Wrote CSV     → {OUT_DIR}/features.csv")

# ---- 10.5 Sanity checks ----
print("\n[Sanity] intent × complexity_bucket:")
(
    final_df
    .group_by("intent", "complexity_bucket")
    .agg(fc.count("*").alias("n"))
    .sort("n", ascending=False)
    .show()
)

print("\n[Sanity] Top 10 longest with has_code=TRUE:")
(
    final_df
    .filter(fc.coalesce("has_code", fc.lit(False)) == fc.lit(True))
    .select("title", "char_len")
    .sort("char_len", ascending=False)
    .limit(10)
    .show()
)

print("✅ Final feature artifacts written with Fenic (no DuckDB).")

INFO:fenic._backends.local.execution:Execution ID: 91025663-31c0-40dc-b167-c39d170ea3d3
INFO:fenic.api.io.writer:Query executed in 61.21ms, returned 0 rows, language model cost: $0.000000, embedding model cost: $0.000000
INFO:fenic._backends.local.execution:Execution ID: 4c7479d0-604b-429a-a287-4dd0ed8dda7d
INFO:fenic.api.io.writer:Query executed in 31.14ms, returned 0 rows, language model cost: $0.000000, embedding model cost: $0.000000
INFO:fenic._backends.local.execution:Execution ID: 23cd866d-42f9-4392-84a3-06b95a67533d
INFO:fenic.api.dataframe.dataframe:Query executed in 13.33ms, returned 14 rows, language model cost: $0.000000, embedding model cost: $0.000000
INFO:fenic._backends.local.execution:Execution ID: e784ce7a-c426-4b23-aa7b-34db9f15ec73
INFO:fenic.api.dataframe.dataframe:Query executed in 11.35ms, returned 10 rows, language model cost: $0.000000, embedding model cost: $0.000000


✅ Wrote Parquet → /content/out/features.parquet
✅ Wrote CSV     → /content/out/features.csv

[Sanity] intent × complexity_bucket:
┌─────────────────────┬───────────────────┬────┐
│ intent              ┆ complexity_bucket ┆ n  │
╞═════════════════════╪═══════════════════╪════╡
│ tutorial/how-to     ┆ long              ┆ 51 │
│ research/explainer  ┆ long              ┆ 42 │
│ case-study/showcase ┆ long              ┆ 24 │
│ opinion/thinkpiece  ┆ long              ┆ 21 │
│ news/announcement   ┆ long              ┆ 16 │
│ …                   ┆ …                 ┆ …  │
│ opinion/thinkpiece  ┆ medium            ┆ 2  │
│ tutorial/how-to     ┆ medium            ┆ 2  │
│ research/explainer  ┆ short             ┆ 1  │
│ opinion/thinkpiece  ┆ short             ┆ 1  │
│ case-study/showcase ┆ medium            ┆ 1  │
└─────────────────────┴───────────────────┴────┘

[Sanity] Top 10 longest with has_code=TRUE:
┌───────────────────────────────────────────────────────────────────────────────────────┬─

## **(Optional) Cluster Report (Exemplars + Optional LLM Summary)**

### **What this cell does**

* **Scores every article against its cluster centroid** using cosine distance on the `emb` vectors you already computed.
* **Picks one exemplar per cluster** (the nearest item to the centroid) and prints a compact table: `cluster`, `cluster_label`, `count`, `exemplar_title`, `exemplar_url`, and `exemplar_dist`.
* (Optional) **Generates a short LLM summary per cluster** from the exemplar and/or top items to create human-friendly labels or blurbs.

### **Why it matters**

* Centroid-nearest examples are a **fast, interpretable proxy** for “what this cluster is about” without reading dozens of items.
* Exemplar tables are great for **sanity-checking clustering quality**, writing slide decks, and **naming clusters**.
* Optional LLM blurbs turn raw clusters into **shareable insights** for non-technical readers.


In [ ]:
# Requires:
# - df_labeled (with: url, title, body_clip, emb, cluster, centroid, cluster_label)
# - df_intent  (with: url, intent)

print("[Report] Building cluster report with exemplars…")

# 1) Pick an exemplar (closest to centroid) per cluster
with_dist = df_labeled.select(
    "*",
    (fc.lit(1.0) - fc.embedding.compute_similarity("emb", "centroid", metric="cosine")).alias("dist_to_centroid")
)

exemplars = (
    with_dist
    .sort(["cluster", "dist_to_centroid", "title"], ascending=[True, True, True])
    .group_by("cluster")
    .agg(
        fc.first("title").alias("exemplar_title"),
        fc.first("url").alias("exemplar_url"),
        fc.first("dist_to_centroid").alias("exemplar_dist"),
        fc.first("body_clip").alias("exemplar_body"),
    )
    .sort("cluster")
)

[Report] Building cluster report with exemplars…


In [ ]:
# 2) Counts + labels
counts = df_labeled.group_by("cluster").agg(fc.count("*").alias("count"))
labels = (
    df_labeled
    .select("cluster", "cluster_label")
    .group_by("cluster")
    .agg(fc.first("cluster_label").alias("cluster_label"))
)

report = (
    exemplars
    .join(counts, on="cluster", how="inner")
    .join(labels, on="cluster", how="inner")
    .select("cluster", "cluster_label", "count", "exemplar_title", "exemplar_url", "exemplar_dist", "exemplar_body")
)

# Simple sanity print
num_clusters = report.select("cluster").group_by("cluster").agg(fc.count("*").alias("n")).count()
print(f"[Report] Rows={df_labeled.count()} | Clusters present: {num_clusters}")
report.select("cluster", "cluster_label", "count", "exemplar_title", "exemplar_url", "exemplar_dist").show(10)

INFO:fenic._backends.local.execution:Execution ID: 9eb20603-75de-46a9-82ac-1f112c2c6082
INFO:fenic._backends.local.execution:Execution ID: febd8cea-0a4d-485b-95fe-359e267f8ffb
INFO:fenic._backends.local.execution:Execution ID: caecc42a-d2a2-45d6-9d73-ac7c0d9284b5


[Report] Rows=185 | Clusters present: 10


INFO:fenic.api.dataframe.dataframe:Query executed in 1013.23ms, returned 10 rows, language model cost: $0.000000, embedding model cost: $0.000000


┌─────────┬─────────────────────┬───────┬─────────────────────┬────────────────────┬───────────────┐
│ cluster ┆ cluster_label       ┆ count ┆ exemplar_title      ┆ exemplar_url       ┆ exemplar_dist │
╞═════════╪═════════════════════╪═══════╪═════════════════════╪════════════════════╪═══════════════╡
│ 6       ┆ The Applications    ┆ 13    ┆ The Applications    ┆ https://towardsdat ┆ 0.241568      │
│         ┆ and Benefits of a   ┆       ┆ and Benefits of a   ┆ ascience.com/the-a ┆               │
│         ┆ PreTrained Model –– ┆       ┆ PreTrained Model –– ┆ pplications-and-be ┆               │
│         ┆ Kaggle’s DogsVSCats ┆       ┆ Kaggle’s DogsVSCats ┆ nefits-of-a-pretra ┆               │
│         ┆                     ┆       ┆                     ┆ ined-model-kaggles ┆               │
│         ┆                     ┆       ┆                     ┆ -dogsvscats-502219 ┆               │
│         ┆                     ┆       ┆                     ┆ 02c696             ┆       

In [ ]:
# 3) Grounded LLM summaries using semantic.extract to Pydantic schema with List[str]
class ClusterBrief(BaseModel):
    summary: List[str] = Field(
        default_factory=list,
        description="1–3 short bullets on the cluster’s focus, audience, and tone."
    )

GUIDE = (
    "You will summarise ONE article snippet to infer the overall CLUSTER focus.\n"
    "- Use 1–3 short bullets (<=30 words each).\n"
    "- No hype; be concrete and technical where relevant.\n"
    "- Mention likely audience (e.g., beginners, practitioners, execs) and tone (tutorial/news/opinion/explainer)."
)

print("[Report] Adding LLM summaries via semantic.extract (grounded)…")

intent_for_exemplar = df_intent.select(
    fc.col("url").alias("exemplar_url"),
    "intent"
)

exemplar_ctx = (
    report
    .join(intent_for_exemplar, on="exemplar_url", how="left")
    .select(
        "cluster", "cluster_label", "count", "exemplar_title", "exemplar_dist", "exemplar_url", "exemplar_body",
        fc.text.concat_ws(
            "\n",
            fc.lit("== TASK =="),
            fc.lit(GUIDE),
            fc.lit("\n== EXEMPLAR TITLE =="),
            fc.col("exemplar_title"),
            fc.lit("\n== INTENT (optional) =="),
            fc.coalesce("intent", fc.lit("unknown")),
            fc.lit("\n== BODY CLIP =="),
            fc.col("exemplar_body")
        ).alias("guided_text")
    )
)

# Extract into a STRUCT column named 'brief' (not a string)
with_brief = exemplar_ctx.select(
    "*",
    fc.semantic.extract("guided_text", ClusterBrief).alias("brief")   # brief.summary is List[str]
)

[Report] Adding LLM summaries via semantic.extract (grounded)…


In [ ]:
# 4) CSV-safe summary: join the List[str] into a single String
summary_str = fc.coalesce(
    fc.text.array_join(fc.col("brief")["summary"], " • "),
    fc.lit("No summary available")
).alias("cluster_summary")

# Build final CSV-friendly frame (only scalar columns)
summarised = (
    with_brief
    .select(
        "cluster",
        "cluster_label",
        "count",
        "exemplar_title",
        "exemplar_url",
        "exemplar_dist",
        summary_str,
    )
    .cache()
)

# 5) Write to CSV
print("[Report] Finalizing cluster report for export…")
OUT_DIR = "/content/out"
summarised.write.csv(f"{OUT_DIR}/cluster_report.csv")
print(f"✅ Saved cluster report → {OUT_DIR}/cluster_report.csv")

INFO:fenic._backends.local.execution:Execution ID: 0bfdf111-7151-49b1-9c1c-503f63a9ab01


[Report] Finalizing cluster report for export…


INFO:fenic._inference.model_client:Creating batch c0e2bebe-3012-46be-b629-7985bae0bec8 with 185 requests for semantic.classify using (model: gpt-4o-mini)
INFO:fenic._inference.model_client:Processing batch c0e2bebe-3012-46be-b629-7985bae0bec8 with 185 requests for semantic.classify using (model: gpt-4o-mini)
Submitting requests for batch: c0e2bebe-3012-46be-b629-7985bae0bec8 (model: gpt-4o-mini): 100%|██████████| 185/185 [00:04<00:00, 45.93req/s, estimated_input_tokens=110079, estimated_output_tokens=11840]
INFO:fenic._inference.model_client:Batch c0e2bebe-3012-46be-b629-7985bae0bec8: Submitted 185 unique requests with Input Tokens: 110079, Output Tokens: 11840, Total Tokens: 121919
Awaiting responses for batch c0e2bebe-3012-46be-b629-7985bae0bec8 (model: gpt-4o-mini): 100%|██████████| 185/185 [00:01<00:00, 157.25res/s]
INFO:fenic._inference.model_client:Batch c0e2bebe-3012-46be-b629-7985bae0bec8: Completed with 185 responses from gpt-4o-mini
INFO:fenic._inference.model_client:Creating

✅ Saved cluster report → /content/out/cluster_report.csv


In [ ]:
# Preview cluster summaries
summarised.select("cluster","cluster_label","count","exemplar_title","cluster_summary").show(10)

INFO:fenic._backends.local.execution:Execution ID: 433c9a50-f4d1-4725-adf2-3f639fe8a0e2
INFO:fenic.api.dataframe.dataframe:Query executed in 2.68ms, returned 10 rows, language model cost: $0.000000, embedding model cost: $0.000000


┌─────────┬──────────────────────────┬───────┬──────────────────────────┬──────────────────────────┐
│ cluster ┆ cluster_label            ┆ count ┆ exemplar_title           ┆ cluster_summary          │
╞═════════╪══════════════════════════╪═══════╪══════════════════════════╪══════════════════════════╡
│ 0       ┆ What is Linear           ┆ 14    ┆ What is Linear           ┆ Explains linear          │
│         ┆ Regression in Machine    ┆       ┆ Regression in Machine    ┆ regression in machine    │
│         ┆ Learning                 ┆       ┆ Learning                 ┆ learning, focusing on    │
│         ┆                          ┆       ┆                          ┆ its definition and       │
│         ┆                          ┆       ┆                          ┆ application. • Targeted  │
│         ┆                          ┆       ┆                          ┆ at practitioners and     │
│         ┆                          ┆       ┆                          ┆ researchers inter

## **(Optional) Recsys Hooks With Centroid Profiles & “More-Like-This (MLT)”**

### **What this cell does**

* **Builds “centroid profiles”**: for every cluster, finds the **Top-N nearest items to that cluster’s centroid** using cosine distance on your `emb` vectors.
* **Exports a flat CSV** of those picks to `/content/out/centroid_profiles.csv` for reporting or downstream use.
* **Adds an ad-hoc “More-Like-This” (MLT) search**: embeds a free-text query once (via OpenAI) and returns the **Top-K most similar items** from `df_labeled`.
* **Optionally exports** the MLT results to `/content/out/mlt_demo.csv`.


### **Why it matters**

* **Fast exemplars per cluster** help you (and stakeholders) see what each cluster is “about” without reading everything.
* **Top-N near centroid** doubles as a solid **recommendations primitive** (good default “featured” items per topic).
* **MLT** turns your corpus into a lightweight **semantic search** surface (great for product discovery, editorial findability, etc.).



In [ ]:
from openai import OpenAI  # only used for optional MLT query embed

# ----  Inputs ----------------------------------------------------
need = {"url", "title", "emb", "cluster", "cluster_label", "centroid"}
missing = need - set(df_labeled.schema.column_names())
assert not missing, f"df_labeled missing {missing}. Re-run Steps 5–6."

items = df_labeled.count()
k_rows = (
    df_labeled
    .select("cluster")
    .filter(fc.col("cluster").is_not_null())
    .sort("cluster")
    .to_pylist()
)
cluster_ids = sorted({row["cluster"] for row in k_rows})
print(f"[Recsys] Items={items} | Clusters={len(cluster_ids)}")

INFO:fenic._backends.local.execution:Execution ID: 908c0cbb-bb57-4470-9b64-4f0930c8ae24
INFO:fenic._backends.local.execution:Execution ID: 954f34e2-43fc-4fc5-b3aa-2bb07a8459ca
INFO:fenic.api.dataframe.dataframe:Query executed in 1078.39ms, returned 185 rows, language model cost: $0.000000, embedding model cost: $0.000000


[Recsys] Items=185 | Clusters=10


In [ ]:
# ---- 1) Centroid profiles: Top-N nearest to each cluster centroid ----------
TOPN_PER_CLUSTER = 5  # tweak as needed

with_centroid_dist = df_labeled.select(
    "*",
    (fc.lit(1.0) - fc.embedding.compute_similarity(
        fc.col("emb"), fc.col("centroid"), metric="cosine"
    )).alias("dist_to_centroid")
)

# Pre-sort once for cheap per-cluster slicing
sorted_by_cluster = with_centroid_dist.sort(
    ["cluster", "dist_to_centroid", "title"], ascending=[True, True, True]
).select("cluster", "cluster_label", "title", "url", "dist_to_centroid")

# Python-side slice: Top-N per cluster
topn_rows = []
for cid in cluster_ids:
    topn_rows.extend(
        sorted_by_cluster
        .filter(fc.col("cluster") == fc.lit(cid))
        .limit(TOPN_PER_CLUSTER)
        .to_pylist()
    )

df_topn = session.create_dataframe(topn_rows)

print("\n[Recsys] Top-N per cluster (closest to centroid):")
df_topn.show(min(len(cluster_ids) * TOPN_PER_CLUSTER, 50))

# Fenic CSV write
centroid_profiles_csv = str(OUT_DIR / "centroid_profiles.csv")
df_topn.write.csv(centroid_profiles_csv)
print(f"✅ Wrote centroid profiles → {centroid_profiles_csv}")

INFO:fenic._backends.local.execution:Execution ID: 90917c8a-a82e-4ac4-93bb-4d0d5975a222
INFO:fenic.api.dataframe.dataframe:Query executed in 622.02ms, returned 5 rows, language model cost: $0.000000, embedding model cost: $0.000000
INFO:fenic._backends.local.execution:Execution ID: bdd7f089-83de-4d11-aa7d-c06a07eb91f6
INFO:fenic.api.dataframe.dataframe:Query executed in 535.81ms, returned 5 rows, language model cost: $0.000000, embedding model cost: $0.000000
INFO:fenic._backends.local.execution:Execution ID: 7431068a-14ac-44d7-bf56-5bb2c7536c47
INFO:fenic.api.dataframe.dataframe:Query executed in 343.98ms, returned 5 rows, language model cost: $0.000000, embedding model cost: $0.000000
INFO:fenic._backends.local.execution:Execution ID: e2d83231-081e-4e82-95b1-b6bf4b019b61
INFO:fenic.api.dataframe.dataframe:Query executed in 338.23ms, returned 5 rows, language model cost: $0.000000, embedding model cost: $0.000000
INFO:fenic._backends.local.execution:Execution ID: 4b1b8a83-6b9f-4f08-96


[Recsys] Top-N per cluster (closest to centroid):
┌─────────┬───────────────────────┬──────────────────────┬──────────────────────┬──────────────────┐
│ cluster ┆ cluster_label         ┆ title                ┆ url                  ┆ dist_to_centroid │
╞═════════╪═══════════════════════╪══════════════════════╪══════════════════════╪══════════════════╡
│ 0       ┆ What is Linear        ┆ What is Linear       ┆ https://medium.com/@ ┆ 0.254008         │
│         ┆ Regression in Machine ┆ Regression in        ┆ mlbot/what-is-linear ┆                  │
│         ┆ Learning              ┆ Machine Learning     ┆ -regression-in-machi ┆                  │
│         ┆                       ┆                      ┆ ne-learning-1d1fd79c ┆                  │
│         ┆                       ┆                      ┆ 0ee1                 ┆                  │
│ 0       ┆ What is Linear        ┆ After liner          ┆ https://medium.com/@ ┆ 0.259559         │
│         ┆ Regression in Machine ┆ regr

In [ ]:
# ---- More-Like-This (MLT) for an ad-hoc query --------------------------

def embed_text_once(text: str) -> list[float]:
    resp = CLIENT.embeddings.create(model=EMB_MODEL, input=text)
    return resp.data[0].embedding

def more_like_this(query_text: str, k: int = 10):
    assert CLIENT is not None, "Set OPENAI_API_KEY to use 'more like this'."
    vec = embed_text_once(query_text)  # plain list[float]

    # Directly compute cosine similarity vs the query vector.
    # No cross-join, no with_column/embedding.from_list needed.
    scored = (
        df_labeled
        .select(
            "title",
            "url",
            "cluster",
            "cluster_label",
            fc.embedding.compute_similarity(
                fc.col("emb"),
                vec,
                metric="cosine"
            ).alias("sim")
        )
        .sort("sim", ascending=False)
        .limit(k)
    )
    return scored

In [ ]:
# Demo query for the recsys
if CLIENT:
    demo_query = "practical transformer tutorials for production ML"
    print(f"\n[MLT] Query → {demo_query!r}")
    mlt = more_like_this(demo_query, k=8)
    mlt.show(8)

    mlt_csv = OUT_DIR / "mlt_demo.csv"

    mlt.write.csv(str(mlt_csv))
    print(f"✅ Wrote MLT results → {mlt_csv}")
else:
    print("\n[MLT] Skipped (no OPENAI_API_KEY set). Set it to enable query-time MLT.")


[MLT] Query → 'practical transformer tutorials for production ML'


INFO:fenic._backends.local.execution:Execution ID: dfc14b05-43f4-41a1-8bc3-8b345fd25138
INFO:fenic.api.dataframe.dataframe:Query executed in 786.36ms, returned 8 rows, language model cost: $0.000000, embedding model cost: $0.000000
INFO:fenic._backends.local.execution:Execution ID: 80fc6b73-0742-48af-bf71-a78f6cab4ba9


┌───────────────────────────┬──────────────────────────┬─────────┬──────────────────────┬──────────┐
│ title                     ┆ url                      ┆ cluster ┆ cluster_label        ┆ sim      │
╞═══════════════════════════╪══════════════════════════╪═════════╪══════════════════════╪══════════╡
│ Getting Started with      ┆ https://medium.com/@mann ┆ 5       ┆ Welcome to the       ┆ 0.41672  │
│ Baselines                 ┆ ingbooks/getting-started ┆         ┆ Meta-World           ┆          │
│                           ┆ -with-baselines-b9324f84 ┆         ┆                      ┆          │
│                           ┆ e183                     ┆         ┆                      ┆          │
│ LinkedIn’s Pro-ML         ┆ https://medium.com/datas ┆ 7       ┆ 20 Questions to Ace  ┆ 0.39419  │
│ Architecture Summarizes   ┆ eries/linkedins-pro-ml-a ┆         ┆ Before Getting a     ┆          │
│ Best Practices for        ┆ rchitecture-summarizes-b ┆         ┆ Machine Learning Job ┆  

INFO:fenic.api.io.writer:Query executed in 1719.87ms, returned 0 rows, language model cost: $0.000000, embedding model cost: $0.000000


✅ Wrote MLT results → /content/out/mlt_demo.csv


# ✅ Conclusion



In this demo we:

* Ingested articles, embedded them, and **clustered** them.
* Selected **centroid-closest exemplars** per cluster to make results human-auditable.
* Used `semantic.extract` with a Pydantic schema to generate **grounded, structured LLM briefs**.
* Produced a **CSV-safe report** by converting list fields to strings (e.g., `text.array_join`) and exporting only scalar columns.

This workflow balances **traceability** (you can always inspect the exemplar and intent) with **automation** (LLM summaries), and keeps outputs BI/CSV-friendly.



# 🧭 Optional TODO Checklist

* [ ] Add `cluster_summary_json` column (raw list as JSON) alongside the joined string.
* [ ] Introduce **outlier detection** (e.g., high `exemplar_dist`) and quarantine bucket.
* [ ] Re-run clustering with different `k` and compare **Silhouette**/**Davies–Bouldin** as a sanity check.
* [ ] Add **deduping** or near-duplicate merging before clustering to reduce noise.
* [ ] Enrich with **source metadata** (author, domain) to improve downstream filters and summaries.